# TMDB Batch Data Ingestion

This notebook handles large-scale ingestion of Movie and TV Show metadata
from TMDB Details API into PostgreSQL Bronze layer.

## Ingestion Strategy
- Stream TMDB Daily ID Exports
- Process IDs in batches
- Retrieve metadata through TMDB Details API
- Preserve raw JSON responses
- Prevent duplicate records
- Track ingestion progress through checkpoints

# Libraries

In [1]:
import importlib
import common

importlib.reload(common)

from common import *

engine = get_engine()


In [2]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT schema_name
        FROM information_schema.schemata
        ORDER BY schema_name;
    """))

    schemas = result.fetchall()

    for schema in schemas:
        print(schema[0])

bronze
gold
information_schema
ops
pg_catalog
pg_toast
public
silver


# making sure if piple is okay

In [3]:
base_url ="https://api.themoviedb.org/3"

movie_export_url = "https://files.tmdb.org/p/exports/movie_ids_{date}.json.gz"
tv_export_url = "https://files.tmdb.org/p/exports/tv_series_ids_{date}.json.gz"

In [4]:
batch_size = 500
request_timeout = 30

print(f'TMBD token: {'confirmed' if TMDB_API_TOKEN else 'missing'}')
print(f'Batch size: {batch_size}')

TMBD token: confirmed
Batch size: 500


In [5]:
export_date = '11_19_2026'

movie_export_url = movie_export_url.format(date=export_date)
tv_export_url = tv_export_url.format(date=export_date)

In [6]:
import gzip

def load_export_ids(url):
    ids = set()
    total = 0
    missing = 0
    duplicates = 0

    with requests.get(url, stream=True, timeout=request_timeout) as response:
        response.raise_for_status()
        response.raw.decode_content = False

        with gzip.GzipFile(fileobj=response.raw) as file:
            for line in file:
                record = json.loads(line)
                tmdb_id = record.get("id")

                total += 1

                if tmdb_id is None:
                    missing += 1
                elif tmdb_id in ids:
                    duplicates += 1
                else:
                    ids.add(tmdb_id)

    return ids, total, missing, duplicates

In [7]:
movie_ids, movie_total, movie_missing, movie_duplicate = load_export_ids(movie_export_url)

tv_ids, tv_total, tv_missing, tv_duplicates = load_export_ids(tv_export_url)

In [8]:
display('MOVIES')
print(f'unique IDs: {len(movie_ids)}')
print(f'Total records: {movie_total}')
print(f'Missing IDs: {movie_missing}')
print(f'Duplicates: {movie_duplicate}')

print('\nTV SHOWS')
print(f'Unique IDs: {len(tv_ids)}')
print(f'Total records: {tv_total}')
print(f'Missing tv shows: {tv_missing}')
print(f'Duplicates: {tv_duplicates}')

'MOVIES'

unique IDs: 1278958
Total records: 1278958
Missing IDs: 0
Duplicates: 0

TV SHOWS
Unique IDs: 237134
Total records: 237134
Missing tv shows: 0
Duplicates: 0


In [9]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS bronze.tmdb_metadata (
            media_type VARCHAR(10) NOT NULL,
            tmdb_id BIGINT NOT NULL,
            payload JSONB NOT NULL,
            ingested_at TIMESTAMPTZ DEFAULT NOW(),
            PRIMARY KEY (media_type, tmdb_id)
        );
    """))

In [10]:
with engine.connect() as conn:
    existing_movie_ids = set(
        conn.execute(text('''
            select tmdb_id
            from bronze.tmdb_metadata
            where media_type = 'movie'   ''')).scalars().all()
    )

    existing_tv_ids = set(
        conn.execute(text('''
            select tmdb_id
            from bronze.tmdb_metadata
            where media_type = 'tv' ''')).scalars().all()
    )

In [11]:
print(f'Existing moves: {len(existing_movie_ids)}')
print(f'Existing tv shows: {len(existing_movie_ids)}')

Existing moves: 102995
Existing tv shows: 102995


In [12]:
new_movie_ids = movie_ids - existing_movie_ids
new_tv_ids = tv_ids - existing_tv_ids

In [13]:
print(f"New Movies: {len(new_movie_ids)}")
print(f"New TV Shows: {len(new_tv_ids)}")
print(f"Total New IDs: {len(new_movie_ids) + len(new_tv_ids)}")

New Movies: 1175963
New TV Shows: 237134
Total New IDs: 1413097


In [14]:
print("Movie IDs:", len(movie_ids))
print("TV IDs:", len(tv_ids))

print("Existing Movie IDs:", len(existing_movie_ids))
print("Existing TV IDs:", len(existing_tv_ids))

print("New Movie IDs:", len(new_movie_ids))
print("New TV IDs:", len(new_tv_ids))

Movie IDs: 1278958
TV IDs: 237134
Existing Movie IDs: 102995
Existing TV IDs: 0
New Movie IDs: 1175963
New TV IDs: 237134


# Testing if working or not

In [15]:
def get_details(tmdb_id, media_type, max_retries=5):
    url = f"{base_url}/{media_type}/{tmdb_id}"
    headers = {
        "Authorization": f"Bearer {TMDB_API_TOKEN}"
    }

    for attempt in range(max_retries):
        try:
            response = requests.get(
                url,
                headers=headers,
                timeout=(15, 60)
            )

            if response.status_code == 200:
                return response.json()

            if response.status_code == 404:
                return None

            if response.status_code == 429 or response.status_code >= 500:
                wait_time = min(2 ** attempt, 60)
                print(f"Retry {attempt + 1}: HTTP {response.status_code}")
                time.sleep(wait_time)
                continue

            response.raise_for_status()

        except requests.RequestException as error:
            if attempt == max_retries - 1:
                raise

            wait_time = min(2 ** attempt, 60)
            print(
                f"Retry {attempt + 1}/{max_retries} "
                f"for ID {tmdb_id}: {error}"
            )
            time.sleep(wait_time)

    raise RuntimeError(
        f"Failed to fetch {media_type} ID {tmdb_id}"
    )

In [16]:
movie_test_id = next(iter(new_movie_ids))

movie_data = get_details(
    movie_test_id,
    "movie"
)

print("Movie ID:", movie_test_id)
print("Data received:", movie_data is not None)

Movie ID: 60518
Data received: False


In [17]:
if movie_data:
    print("Title:", movie_data.get("title"))
    print("TMDB ID:", movie_data.get("id"))
    print("Fields:", len(movie_data))

In [18]:
tv_test_id = next(iter(new_tv_ids))

tv_data = get_details(
    tv_test_id,
    "tv"
)

print("TV ID:", tv_test_id)
print("Data received:", tv_data is not None)

TV ID: 1
Data received: True


In [19]:
if tv_data:
    print("Name:", tv_data.get("name"))
    print("TMDB ID:", tv_data.get("id"))
    print("Fields:", len(tv_data))

Name: Pride
TMDB ID: 1
Fields: 33


# Data taking

## movie

In [20]:
EXPORT_DATE_DB = "2026-11-19"

with engine.connect() as conn:
    movie_checkpoint = conn.execute(text("""
        SELECT
            checkpoint_id,
            source,
            media_type,
            export_date,
            last_processed_id,
            records_processed,
            last_processed_at,
            updated_at
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND media_type = 'movie'
          AND export_date = :export_date;
    """), {
        "export_date": EXPORT_DATE_DB
    }).fetchone()

print(movie_checkpoint)

(10, 'tmdb', 'movie', datetime.date(2026, 11, 19), 172167, 84995, datetime.datetime(2026, 11, 22, 3, 10, 53, 328180, tzinfo=datetime.timezone.utc), datetime.datetime(2026, 11, 22, 3, 10, 53, 328180, tzinfo=datetime.timezone.utc))


In [25]:
with engine.connect() as conn:

    existing_movie_ids = set(
        conn.execute(text("""
            SELECT tmdb_id
            FROM bronze.tmdb_metadata
            WHERE media_type = 'movie'
        """)).scalars().all()
    )

    last_processed_id = conn.execute(text("""
        SELECT last_processed_id
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND media_type = 'movie'
          AND export_date = :export_date;
    """), {
        "export_date": EXPORT_DATE_DB
    }).scalar_one()

new_movie_ids = sorted(
    tmdb_id
    for tmdb_id in movie_ids
    if tmdb_id not in existing_movie_ids
    and (
        last_processed_id is None
        or tmdb_id > last_processed_id
    )
)

print(f"Existing Movies: {len(existing_movie_ids):,}")
print(f"Last processed ID: {last_processed_id}")
print(f"Remaining Movies: {len(new_movie_ids):,}")

Existing Movies: 157,993
Last processed ID: 270990
Remaining Movies: 1,120,958


In [22]:
batch_size = 1000

for i in range(0, len(new_movie_ids), batch_size):

    batch_ids = new_movie_ids[i:i + batch_size]
    records = []

    # Fetch movie details
    for tmdb_id in batch_ids:

        data = get_details(tmdb_id, "movie")

        if data is not None:
            records.append({
                "media_type": "movie",
                "tmdb_id": tmdb_id,
                "payload": json.dumps(data)
            })

        time.sleep(0.05)

    # Save records and update checkpoint
    with engine.begin() as conn:

        if records:
            conn.execute(
                text("""
                    INSERT INTO bronze.tmdb_metadata
                        (media_type, tmdb_id, payload)
                    VALUES
                        (:media_type, :tmdb_id, CAST(:payload AS JSONB))
                    ON CONFLICT (media_type, tmdb_id)
                    DO NOTHING
                """),
                records
            )

        conn.execute(
            text("""
                UPDATE ops.ingestion_checkpoints
                SET
                    last_processed_id = :last_id,
                    records_processed = records_processed + :processed,
                    last_processed_at = NOW(),
                    updated_at = NOW()
                WHERE source = 'tmdb'
                  AND media_type = 'movie'
                  AND export_date = :export_date
            """),
            {
                "last_id": batch_ids[-1],
                "processed": len(records),
                "export_date": EXPORT_DATE_DB
            }
        )

    # Progress
    print(f"Movie batch {i // batch_size + 1} completed")
    print(f"IDs processed: {len(batch_ids):,}")
    print(f"Records fetched: {len(records):,}")
    print(f"Last processed ID: {batch_ids[-1]:,}")
    print("-" * 40)

Movie batch 1 completed
IDs processed: 1,000
Records fetched: 1,000
Last processed ID: 174,168
----------------------------------------
Movie batch 2 completed
IDs processed: 1,000
Records fetched: 1,000
Last processed ID: 176,015
----------------------------------------
Movie batch 3 completed
IDs processed: 1,000
Records fetched: 1,000
Last processed ID: 178,594
----------------------------------------
Movie batch 4 completed
IDs processed: 1,000
Records fetched: 1,000
Last processed ID: 181,130
----------------------------------------
Movie batch 5 completed
IDs processed: 1,000
Records fetched: 1,000
Last processed ID: 183,707
----------------------------------------
Movie batch 6 completed
IDs processed: 1,000
Records fetched: 1,000
Last processed ID: 185,648
----------------------------------------
Movie batch 7 completed
IDs processed: 1,000
Records fetched: 1,000
Last processed ID: 187,644
----------------------------------------
Movie batch 8 completed
IDs processed: 1,000
Rec

KeyboardInterrupt: 

In [23]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT
            media_type,
            last_processed_id,
            records_processed,
            last_processed_at
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND export_date = :export_date
        ORDER BY media_type;
    """), {"export_date": EXPORT_DATE_DB})

    for row in result:
        print(row)

('movie', 270990, 139993, datetime.datetime(2026, 11, 24, 0, 8, 1, 752595, tzinfo=datetime.timezone.utc))
('tv', None, 0, None)


In [24]:
with engine.connect() as conn:

    result = conn.execute(text("""
        SELECT
            media_type,
            COUNT(*) AS total_records,
            MIN(tmdb_id) AS min_id,
            MAX(tmdb_id) AS max_id
        FROM bronze.tmdb_metadata
        GROUP BY media_type
        ORDER BY media_type;
    """))

    for row in result:
        print(row)

('movie', 157993, 2, 270990)


## tv

In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO ops.ingestion_checkpoints (
            source,
            media_type,
            export_date,
            last_processed_id,
            records_processed
        )
        VALUES (
            'tmdb',
            'tv',
            :export_date,
            NULL,
            0
        )
        ON CONFLICT DO NOTHING;
    """), {
        "export_date": EXPORT_DATE_DB
    })

In [ ]:
with engine.connect() as conn:
    checkpoint = conn.execute(text("""
        SELECT *
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND media_type = 'tv'
          AND export_date = :export_date;
    """), {
        "export_date": EXPORT_DATE_DB
    }).fetchone()

print(checkpoint)

(11, 'tmdb', 'tv', datetime.date(2026, 11, 19), None, 0, None, None, datetime.datetime(2026, 11, 20, 20, 50, 8, 618447, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')), datetime.datetime(2026, 11, 20, 20, 50, 8, 618447, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC')))


In [ ]:
with engine.connect() as conn:
    last_processed_id = conn.execute(text("""
        SELECT last_processed_id
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND media_type = 'tv'
          AND export_date = :export_date;
    """), {
        "export_date": EXPORT_DATE_DB
    }).scalar()

print(f"Last processed ID: {last_processed_id}")

Last processed ID: None


In [ ]:
with engine.connect() as conn:
    existing_tv_ids = set(
        conn.execute(text("""
            SELECT tmdb_id
            FROM bronze.tmdb_metadata
            WHERE media_type = 'tv'
        """)).scalars().all()
    )

new_tv_ids = sorted(
    tmdb_id
    for tmdb_id in tv_ids
    if tmdb_id not in existing_tv_ids
    and (last_processed_id is None or tmdb_id > last_processed_id)
)

print(f"Remaining TV Shows: {len(new_tv_ids):,}")

In [ ]:
batch_size = 1000

for i in range(0, len(new_tv_ids), batch_size):

    batch_ids = new_tv_ids[i:i + batch_size]
    records = []

    for tmdb_id in batch_ids:

        data = get_details(tmdb_id, "tv")

        if data is not None:
            records.append({
                "media_type": "tv",
                "tmdb_id": tmdb_id,
                "payload": json.dumps(data)
            })

        time.sleep(0.05)

    with engine.begin() as conn:

        if records:
            conn.execute(
                text("""
                    INSERT INTO bronze.tmdb_metadata
                        (media_type, tmdb_id, payload)
                    VALUES
                        (:media_type, :tmdb_id, CAST(:payload AS JSONB))
                    ON CONFLICT (media_type, tmdb_id)
                    DO NOTHING
                """),
                records
            )

        conn.execute(
            text("""
                UPDATE ops.ingestion_checkpoints
                SET
                    last_processed_id = :last_id,
                    records_processed = records_processed + :processed,
                    last_processed_at = NOW(),
                    updated_at = NOW()
                WHERE source = 'tmdb'
                  AND media_type = 'tv'
                  AND export_date = :export_date
            """),
            {
                "last_id": batch_ids[-1],
                "processed": len(records),
                "export_date": EXPORT_DATE_DB
            }
        )

    print(f"TV batch {i // batch_size + 1} completed")
    print(f"IDs processed: {len(batch_ids):,}")
    print(f"Records fetched: {len(records):,}")
    print(f"Last processed ID: {batch_ids[-1]:,}")
    print("-" * 40)